# Triage Pilot — Phase 0 Checkpoint 4: Gemma 4 E4B Baseline

**Purpose:** Zero-shot Gemma 4 E4B across the frozen test set for all four SAM tasks. Produces the accuracy / F1 / latency / VRAM numbers that SAMs will be measured against.

**Runs on:** Vertex AI with A100 (preferred) or L4. E4B is too large for a T4 in FP16 without heavy quantization — use Vertex for this one.

**Output:** writes results into `baselines/baseline_results.json` under the key `gemma_e4b`.

## 1. Config

In [ ]:
import os

# Adjust if running on Vertex with a different mount
TRIAGE_ROOT = '/content/drive/MyDrive/setique/triage'
DATA_PROCESSED_DIR = f'{TRIAGE_ROOT}/data/processed'
EVAL_DIR = f'{TRIAGE_ROOT}/eval'
BASELINES_DIR = f'{TRIAGE_ROOT}/baselines'
PROMPTS_DIR = f'{BASELINES_DIR}/prompts'

TEST_PATH = f'{DATA_PROCESSED_DIR}/test.jsonl'
HASH_PATH = f'{EVAL_DIR}/test_set_hash.txt'
RESULTS_PATH = f'{BASELINES_DIR}/baseline_results.json'

MODEL_KEY = 'gemma_e4b'
# Canonical HuggingFace ID for Gemma 4 E4B instruction-tuned.
# Capital E in E4B, lowercase 'it' suffix. Gated — see preflight cell.
MODEL_ID = 'google/gemma-4-E4B-it'

# Set to a small int for a smoke test run; None for full test set
LIMIT = None

## 2. Mount + imports

In [ ]:
# Drive mount (comment out if running on Vertex with a different filesystem)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    print('Not running in Colab — assuming TRIAGE_ROOT is already accessible.')

!pip install -q transformers accelerate bitsandbytes scikit-learn

In [ ]:
import os, sys

# Make baselines/runner.py importable regardless of runtime:
#   - Colab / Vertex: clone the triage repo into /content/triage (or pull updates).
#   - Local VS Code kernel launched from repo root: reuse cwd.
# Prompts are read from the repo clone (code, not data). Test data stays on
# Drive (too big for the repo) — TEST_PATH / HASH_PATH / RESULTS_PATH unchanged.
if os.path.exists('/content'):
    REPO_ROOT = '/content/triage'
    if not os.path.exists(REPO_ROOT):
        !git clone https://github.com/iixiiartist/triage.git {REPO_ROOT}
    else:
        !cd {REPO_ROOT} && git pull --ff-only
else:
    REPO_ROOT = os.getcwd()

sys.path.insert(0, REPO_ROOT)

# Prompts live in the repo, not on Drive — override the placeholder from cell 1
PROMPTS_DIR = f'{REPO_ROOT}/baselines/prompts'
print(f'REPO_ROOT   = {REPO_ROOT}')
print(f'PROMPTS_DIR = {PROMPTS_DIR}')

from baselines.runner import (
    verify_test_hash, load_test_set, load_prompt,
    run_inference, merge_results,
    peak_vram_gb, reset_vram_tracking,
)

## Preflight — HF auth, model existence, frozen-set hash

Three cheap checks that run before model weights start downloading. Each catches a failure mode that would otherwise burn 5+ minutes of compute before surfacing:

- **HF auth.** Gemma 4 is gated. The `login()` cell only works if the "Agree and access repository" button has been clicked on the model page in a browser.
- **Model existence.** `HfApi.model_info(MODEL_ID)` catches typos and access issues without pulling weights.
- **Expected hash.** Hardcoded comparison against the committed `e794b4c…` hash — belt-and-suspenders vs. the full hash verification in section 3.

Fail loudly here; don't continue to model load if any of the three trip.

In [ ]:
# Preflight 1: HuggingFace login
# Gemma 4 is GATED — you MUST click "Agree and access repository" at
# https://huggingface.co/google/gemma-4-E4B-it (and E2B-it) in a browser before
# this notebook can download the weights. The token prompt will appear below.
# DO NOT paste the token into this cell's source. Uncheck "Add token as git
# credential?" when asked so the token doesn't leak into git config.
from huggingface_hub import login
login()

In [ ]:
# Preflight 2: model existence + gated-access probe
# Catches MODEL_ID typos and gated-access issues before we spend 5+ minutes
# downloading weights. Success prints canonical id / gated flag / last_modified.
# Failure re-raises so the notebook doesn't silently continue to the load step.
from huggingface_hub import HfApi

_api = HfApi()
try:
    _info = _api.model_info(MODEL_ID)
    print(f'  model_id:      {_info.modelId}')
    print(f'  gated:         {_info.gated}')
    print(f'  last_modified: {_info.last_modified}')
    print('  → model reachable; OK to proceed')
except Exception as _e:
    print(f'  [{type(_e).__name__}] {_e}')
    print('  → fix MODEL_ID or accept gated terms on the HF model page, then retry')
    raise

In [ ]:
# Preflight 3: expected frozen-set hash (belt-and-suspenders)
# Hardcoded below is the hash committed in 2026-04-23. This cell catches wrong
# branch / stale Drive / tampered hash file BEFORE we load the model. The
# runner's verify_test_hash() will also recompute the file hash during section
# 3 — this cell is the cheap early check.
EXPECTED_TEST_HASH = 'e794b4c399a128634248747ad67b713aac933673871f19e649f3ed50989392ca'

with open(HASH_PATH) as _f:
    _committed_hash = _f.read().strip().split()[0]

if _committed_hash != EXPECTED_TEST_HASH:
    raise RuntimeError(
        f'Frozen-test hash mismatch — refusing to run baseline.\n'
        f'  expected: {EXPECTED_TEST_HASH}\n'
        f'  found:    {_committed_hash} (from {HASH_PATH})\n'
        'Check the checked-out branch, Drive sync, or eval/test_set_hash.txt.'
    )
print(f'Frozen-test hash matches committed baseline: {EXPECTED_TEST_HASH[:16]}...')

## 3. Verify frozen test set

In [ ]:
test_hash = verify_test_hash(TEST_PATH, HASH_PATH)
records = load_test_set(TEST_PATH)
print(f'Test hash verified: {test_hash}')
print(f'Loaded {len(records)} test records')

## 4. Load Gemma 4 E4B

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForCausalLM

reset_vram_tracking()

# AutoProcessor (not AutoTokenizer) per the Gemma 4 model card — Gemma 4 is a
# multimodal family (text + image + audio). For text-only classification
# AutoProcessor still works and is the canonical abstraction.
processor = AutoProcessor.from_pretrained(MODEL_ID)
# Prefer bf16 on A100/L4; fall back to fp16 if the device doesn't support it
preferred_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=preferred_dtype,
    device_map='auto',
)
model.eval()
print(f'Loaded {MODEL_ID} in {preferred_dtype}. Peak VRAM after load: {peak_vram_gb():.2f} GB')

## 5. Build the generate function

Greedy decode with a tight max_new_tokens — all four SAMs produce short label outputs, so we don't need sampling.

In [ ]:
@torch.inference_mode()
def generate(prompt: str, max_new_tokens: int = 16) -> str:
    """
    Gemma 4 zero-shot inference with thinking mode disabled.

    Critical flag:
      enable_thinking=False — suppresses the <|think|> reasoning prelude.
      Without this, max_new_tokens=16 would be consumed by reasoning tokens
      before a label appears, and latency measurements would be meaningless.

    Output parsing:
      Gemma 4 E2B/E4B emit empty <|channel>thought\\n<|channel|>[answer]
      wrappers around the answer even with thinking disabled. parse_response()
      on the processor handles this cleanly. Defensive fallback to
      skip_special_tokens=True decode if parse_response returns an unexpected
      shape (e.g., a long string, embedded channel tokens, or multiple paragraphs).
    """
    messages = [{'role': 'user', 'content': prompt}]
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = processor(text=text, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[-1]

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=processor.tokenizer.eos_token_id,
    )
    new_tokens = out[0][input_len:]

    raw = processor.decode(new_tokens, skip_special_tokens=False)
    if hasattr(processor, 'parse_response'):
        parsed = processor.parse_response(raw)
        candidate = parsed.get('response', raw) if isinstance(parsed, dict) else str(parsed)
        # Defensive: parse_response should yield a short label-like string.
        # If it gave us something weird (long, channel tokens, paragraph breaks),
        # log + fall back to plain decode so the probe can show us the true shape.
        if len(candidate) > 100 or '<|' in candidate or '\n\n' in candidate:
            print(f'WARN: parse_response returned unexpected shape: {candidate!r}')
            return processor.decode(new_tokens, skip_special_tokens=True)
        return candidate
    return processor.decode(new_tokens, skip_special_tokens=True)

print('Smoke test output:', generate('Respond with the single word: ready'))

## Probe — verify output format before the full run

Three generate() calls across three SAM prompts (lang, intent, route) with `max_new_tokens=500` so a runaway thinking prelude would be visible instead of truncated. Different label vocabularies may trigger different decode behaviors, so one probe per task.

Stop and re-investigate if any of these appear in the raw output:
- `<|channel>`, `<|think|>`, or other `<|...|>` tokens leaking through `parse_response`
- Literal `thought\n` prefix before the label
- Multi-line reasoning followed by the label (max_new_tokens in the full run would truncate this)

If the three probes come back clean and label-like, proceed to the full run.

In [ ]:
# Probe: verify output format on three prompts before the ~2h full baseline.
# One per SAM task with different label vocabularies to catch task-specific
# decode surprises. Runs three generate() calls with max_new_tokens=500 (so a
# runaway thinking prelude would be visible instead of truncated).
#
# Looking for: (a) short, label-like output, (b) NO leaking <|channel> or
# literal 'thought' tokens in the decoded string. If either shows up, stop
# and revisit generate() / parse_label() before committing to the full run.
for _sam_name in ['sam_lang', 'sam_intent', 'sam_route']:
    _probe_prompt = build_user_message(
        load_prompt(f'{PROMPTS_DIR}/{_sam_name}.txt'),
        records[0]['text'],
    )
    _raw = generate(_probe_prompt, max_new_tokens=500)
    print(f'=== {_sam_name} RAW PROBE ===')
    print(repr(_raw))
    print(f'=== length: {len(_raw)} ===')
    print()

## 6. Run all four SAM tasks

Each task reuses the same frozen test records, scored against a different `target_field` and a different frozen prompt.

In [ ]:
# Label sets (mirror what's in the eval freeze notebook)
LANG_CODES = {'en','es','fr','de','pt','it','nl','ja','zh','ko',
              'ar','ru','pl','tr','sv','hi','id','vi','th','other'}
INTENT_LABELS = {
    'billing_question','billing_dispute','refund_request','cancel_subscription',
    'password_reset','account_access','update_profile','delete_account',
    'how_to','feature_request','bug_report','compatibility_question',
    'order_status','shipping_question','return_request','damaged_item',
    'technical_issue','escalation_request','complaint','compliment',
    'spam','unclear','other','out_of_scope',
}
PRIORITY_LABELS = {'P1','P2','P3','P4'}
ROUTE_LABELS = {
    'billing_team','technical_support','account_security',
    'returns_and_shipping','product_feedback','legal_compliance',
    'human_escalation','auto_close',
}

TASKS = [
    ('sam_lang',     'lang',     LANG_CODES,     'other'),
    ('sam_intent',   'intent',   INTENT_LABELS,  'other'),
    ('sam_priority', 'priority', PRIORITY_LABELS,'P3'),
    ('sam_route',    'route',    ROUTE_LABELS,   'human_escalation'),
]

results_per_sam = {}
for sam_name, target_field, valid_labels, fallback in TASKS:
    print(f'\n=== {sam_name} ===')
    prompt_path = f'{PROMPTS_DIR}/{sam_name}.txt'
    prompt_template = load_prompt(prompt_path)
    reset_vram_tracking()
    result = run_inference(
        records=records,
        prompt_template=prompt_template,
        generate_fn=generate,
        valid_labels=valid_labels,
        target_field=target_field,
        fallback=fallback,
        limit=LIMIT,
        verbose=True,
    )
    result['peak_vram_gb'] = peak_vram_gb()
    results_per_sam[sam_name] = result
    print(f'  accuracy:   {result["accuracy"]:.4f}')
    print(f'  f1_macro:   {result["f1_macro"]:.4f}')
    print(f'  latency p50:{result["latency_ms_p50"]:.0f} ms')
    print(f'  latency p95:{result["latency_ms_p95"]:.0f} ms')
    print(f'  fallback %: {result["parse_fallback_rate"]*100:.2f}%')
    print(f'  peak VRAM:  {result["peak_vram_gb"]:.2f} GB')

## 7. Persist results

In [ ]:
from datetime import datetime

metadata = {
    'model_id': MODEL_ID,
    'dtype': str(preferred_dtype).split('.')[-1],
    'test_hash': test_hash,
    'n_test': len(records),
    'limit': LIMIT,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
    'decode': 'greedy, max_new_tokens=16',
}

merge_results(RESULTS_PATH, MODEL_KEY, results_per_sam, metadata)
print(f'Results merged into {RESULTS_PATH} under key "{MODEL_KEY}"')

## 8. Cost estimate

Rough per-1K-inference cost on whatever instance you're using. Update the hourly rate to match your Vertex instance.

In [ ]:
# Edit to match your actual Vertex instance hourly cost
INSTANCE_HOURLY_USD = 3.67  # placeholder for A100 40GB

for sam_name, res in results_per_sam.items():
    mean_sec = res['latency_ms_mean'] / 1000.0
    cost_per_1k = mean_sec * 1000 / 3600 * INSTANCE_HOURLY_USD
    print(f'{sam_name:14s}  ${cost_per_1k:.4f} per 1K inferences')

## 9. Checkpoint 4 verdict for E4B

- [ ] All four SAMs have recorded metrics
- [ ] `parse_fallback_rate` is reasonable (<10% ideally; >25% means the prompt is not working)
- [ ] `baseline_results.json` committed to the repo with the `gemma_e4b` key populated
- [ ] Run the E2B baseline notebook next to populate `gemma_e2b`